In [1]:
import pandas as pd
import numpy as np

In [2]:

df_energy = pd.read_csv("energy_data_example/data/energy_data_2020_2026.csv")


In [3]:

(df_energy.head())


,date,year,month,day,hour,weekday,load,residual_load,comment
0,2020-01-01,2020,1,1,1,Mittwoch,43.50050,36.89025,NaN
1,2020-01-01,2020,1,1,2,Mittwoch,42.59875,35.55475,NaN
2,2020-01-01,2020,1,1,3,Mittwoch,41.46375,34.03350,NaN
3,2020-01-01,2020,1,1,4,Mittwoch,40.27950,32.73475,NaN
4,2020-01-01,2020,1,1,5,Mittwoch,40.26475,32.51575,NaN


In [4]:
df_energy['hour_0'] = df_energy['hour'] - 1
df_energy


,date,year,month,day,hour,weekday,load,residual_load,comment,hour_0
0,2020-01-01,2020,1,1,1,Mittwoch,43.50050,36.89025,NaN,0
1,2020-01-01,2020,1,1,2,Mittwoch,42.59875,35.55475,NaN,1
2,2020-01-01,2020,1,1,3,Mittwoch,41.46375,34.03350,NaN,2
3,2020-01-01,2020,1,1,4,Mittwoch,40.27950,32.73475,NaN,3
4,2020-01-01,2020,1,1,5,Mittwoch,40.26475,32.51575,NaN,4
...,...,...,...,...,...,...,...,...,...,...
59298,2026-10-06,2026,10,6,20,Dienstag,60.19978,59.50861,NaN,19
59299,2026-10-06,2026,10,6,21,Dienstag,58.31624,57.33501,NaN,20
59300,2026-10-06,2026,10,6,22,Dienstag,54.29507,52.86135,NaN,21
59301,2026-10-06,2026,10,6,23,Dienstag,50.70071,48.84443,NaN,22


In [5]:

# Create naive datetime
df_energy['dt_naive'] = pd.to_datetime(df_energy[['year', 'month', 'day']]) + pd.to_timedelta(df_energy['hour_0'], unit='h')

# Localize to German time (this handles CEST/CET automatically)
df_energy['dt_local'] = df_energy['dt_naive'].dt.tz_localize('Europe/Berlin', ambiguous='NaT', nonexistent='NaT')

# Convert to UTC to match DWD
df_energy['dt_utc'] = df_energy['dt_local'].dt.tz_convert('UTC')

# Select relevant columns
df_energy_clean = df_energy[['dt_utc', 'load', 'residual_load']].dropna(subset=['dt_utc']).copy()
(df_energy_clean.head())

,dt_utc,load,residual_load
0,2019-12-31 23:00:00+00:00,43.50050,36.89025
1,2020-01-01 00:00:00+00:00,42.59875,35.55475
2,2020-01-01 01:00:00+00:00,41.46375,34.03350
3,2020-01-01 02:00:00+00:00,40.27950,32.73475
4,2020-01-01 03:00:00+00:00,40.26475,32.51575


In [6]:
def load_dwd_txt(filepath, value_cols):
    # Read with semicolon separator, skip initial space to handle leading spaces
    df = pd.read_csv(filepath, sep=';', skipinitialspace=True)
    
    # Strip column names
    df.columns = df.columns.str.strip()
    
    # Parse MESS_DATUM (YYYYMMDDHH) to datetime
    # It's a string/integer, convert to string first
    df['dt_utc'] = pd.to_datetime(df['MESS_DATUM'].astype(str), format='%Y%m%d%H', utc=True)
    
    # Handle missing values (-999)
    for col in value_cols:
        # Ensure column is numeric (strip spaces if any)
        if df[col].dtype == object:
            df[col] = pd.to_numeric(df[col].str.strip(), errors='coerce')
        df[col] = df[col].replace(-999, np.nan)
        
    return df[['dt_utc'] + value_cols]


In [7]:

# Load Wind (F = mean wind speed in m/s)
df_wind = load_dwd_txt("wind_data_example\produkt_ff_stunde_20250418_20261019_01420.txt", ['F'])
# Rename to match slides convention
df_wind = df_wind.rename(columns={'F': 'WS10'})

# Load Temperature (TT_TU = air temp at 2m in deg C)
df_temp = load_dwd_txt("air_temp_data_example\produkt_tu_stunde_20250418_20261019_01420.txt", ['TT_TU'])
df_temp = df_temp.rename(columns={'TT_TU': 'T2M'})



<>:2: SyntaxWarning: invalid escape sequence '\p'
<>:7: SyntaxWarning: invalid escape sequence '\p'
<>:2: SyntaxWarning: invalid escape sequence '\p'
<>:7: SyntaxWarning: invalid escape sequence '\p'
C:\Users\Abhishek Bajpai\AppData\Local\Temp\ipykernel_13760\567088005.py:2: SyntaxWarning: invalid escape sequence '\p'
  df_wind = load_dwd_txt("wind_data_example\produkt_ff_stunde_20250418_20261019_01420.txt", ['F'])
C:\Users\Abhishek Bajpai\AppData\Local\Temp\ipykernel_13760\567088005.py:7: SyntaxWarning: invalid escape sequence '\p'
  df_temp = load_dwd_txt("air_temp_data_example\produkt_tu_stunde_20250418_20261019_01420.txt", ['TT_TU'])


In [8]:
df_temp

,dt_utc,T2M
0,2025-04-18 00:00:00+00:00,8.4
1,2025-04-18 01:00:00+00:00,8.0
2,2025-04-18 02:00:00+00:00,8.0
3,2025-04-18 03:00:00+00:00,7.9
4,2025-04-18 04:00:00+00:00,7.6
...,...,...
13195,2026-10-19 19:00:00+00:00,10.5
13196,2026-10-19 20:00:00+00:00,10.3
13197,2026-10-19 21:00:00+00:00,9.9
13198,2026-10-19 22:00:00+00:00,9.0


In [9]:
df_wind

,dt_utc,WS10
0,2025-04-18 00:00:00+00:00,2.5
1,2025-04-18 01:00:00+00:00,3.4
2,2025-04-18 02:00:00+00:00,3.3
3,2025-04-18 03:00:00+00:00,3.3
4,2025-04-18 04:00:00+00:00,3.4
...,...,...
13085,2026-10-19 19:00:00+00:00,2.2
13086,2026-10-19 20:00:00+00:00,2.9
13087,2026-10-19 21:00:00+00:00,3.4
13088,2026-10-19 22:00:00+00:00,3.9


In [10]:
df_weather = pd.merge(df_temp, df_wind, on='dt_utc', how='outer').sort_values('dt_utc')

df_master = pd.merge(df_weather, df_energy_clean, on='dt_utc', how='outer').sort_values('dt_utc')

df_master = df_master.set_index('dt_utc')


In [11]:

(df_master.head())


,T2M,WS10,load,residual_load
dt_utc,,,,
2019-12-31 23:00:00+00:00,NaN,NaN,43.50050,36.89025
2020-01-01 00:00:00+00:00,NaN,NaN,42.59875,35.55475
2020-01-01 01:00:00+00:00,NaN,NaN,41.46375,34.03350
2020-01-01 02:00:00+00:00,NaN,NaN,40.27950,32.73475
2020-01-01 03:00:00+00:00,NaN,NaN,40.26475,32.51575


In [12]:
(df_master.tail())


,T2M,WS10,load,residual_load
dt_utc,,,,
2026-10-19 19:00:00+00:00,10.5,2.2,NaN,NaN
2026-10-19 20:00:00+00:00,10.3,2.9,NaN,NaN
2026-10-19 21:00:00+00:00,9.9,3.4,NaN,NaN
2026-10-19 22:00:00+00:00,9.0,3.9,NaN,NaN
2026-10-19 23:00:00+00:00,8.0,3.5,NaN,NaN


In [13]:
(df_master.info())


<class 'pandas.DataFrame'>
DatetimeIndex: 59607 entries, 2019-12-31 23:00:00+00:00 to 2026-10-19 23:00:00+00:00
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   T2M            13132 non-null  float64
 1   WS10           13079 non-null  float64
 2   load           59291 non-null  float64
 3   residual_load  59291 non-null  float64
dtypes: float64(4)
memory usage: 2.3 MB


In [14]:
(df_master.isna().sum())

T2M              46475
WS10             46528
load               316
residual_load      316
dtype: int64

In [15]:
df_master.to_csv("historical_master_data.csv")